# Practical Exam: Customer Purchase Prediction

RetailTech Solutions is a fast-growing international e-commerce platform operating in over 20 countries across Europe, North America, and Asia. They specialize in fashion, electronics, and home goods, with a unique business model that combines traditional retail with a marketplace for independent sellers.

The company has seen rapid growth. A key part of their success has been their data-driven approach to personalization. However, as they plan their expansion into new markets, they need to improve their ability to predict customer behavior.

Their marketing team wants to predict which customers are most likely to make a purchase based on their browsing behavior.

As an AI Engineer, you will help build this prediction system. Your work will directly impact RetailTech's growth strategy and their goal of increasing revenue.


## Data Description

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

# Task 1

The marketing team has collected customer session data in `raw_customer_data.csv`, but it contains missing values and inconsistencies that need to be addressed.
Create a cleaned version of the dataframe:

- Start with the data in the file `raw_customer_data.csv`
- Your output should be a DataFrame named `clean_data`
- All column names and values should match the table below.
</br>

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

In [8]:
# Write your answer to Task 1 here 

import pandas as pd

clean_data = pd.read_csv('raw_customer_data.csv')

clean_data['time_spent'] = clean_data['time_spent'].fillna(clean_data['time_spent'].median())
clean_data['pages_viewed'] = clean_data['pages_viewed'].fillna(clean_data['pages_viewed'].mean())
clean_data['basket_value'] = clean_data['basket_value'].fillna(0)
clean_data['device_type'] = clean_data['device_type'].fillna('Unknown')
clean_data['customer_type'] = clean_data['customer_type'].fillna('New')

clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    500 non-null    int64  
 1   time_spent     500 non-null    float64
 2   pages_viewed   500 non-null    float64
 3   basket_value   500 non-null    float64
 4   device_type    500 non-null    object 
 5   customer_type  500 non-null    object 
 6   purchase       500 non-null    int64  
dtypes: float64(3), int64(2), object(2)
memory usage: 27.5+ KB


# Task 2
The pre-cleaned dataset `model_data.csv` needs to be prepared for our neural network.
Create the model features:

- Start with the data in the file `model_data.csv`
- Scale numerical features (`time_spent`, `pages_viewed`, `basket_value`) to 0-1 range
- Apply one-hot encoding to the categorical features (`device_type`, `customer_type`)
    - The column names should have the following format: variable_name_category_name (e.g., `device_type_Desktop`)
- Your output should be a DataFrame named `model_feature_set`, with all column names from `model_data.csv` except for the columns where one-hot encoding was applied.


In [31]:
# Write your answer to Task 2 here

from sklearn.preprocessing import MinMaxScaler

model_feature_set = pd.read_csv('model_data.csv')

model_feature_set_device = pd.get_dummies(model_feature_set["device_type"], drop_first=False)
model_feature_set_cus = pd.get_dummies(model_feature_set["customer_type"], drop_first=False)

model_feature_set_device = model_feature_set_device.rename(columns={
    'Desktop':'device_type_Desktop',
    'Mobile':'device_type_Mobile',
    'Unknown':'device_type_Unknown',
    'Tablet':'device_type_Tablet',
})
model_feature_set_cus = model_feature_set_cus.rename(columns={
    'New':'customer_type_New',
    'Returning':'customer_type_Returning',
})

model_feature_set_n = model_feature_set.drop(columns=['device_type', 'customer_type', 'purchase', 'customer_id'], axis=1)

scaler = MinMaxScaler()
model_feature_set_n[['time_spent', 'pages_viewed', 'basket_value']] = scaler.fit_transform(model_feature_set_n[['time_spent', 'pages_viewed', 'basket_value']])

# combine
model_feature_set =  pd.concat(
    [
        model_feature_set['customer_id'],
        model_feature_set_n,
        model_feature_set_device,
        model_feature_set_cus,
        model_feature_set['purchase'],
    ],
    axis=1,
)

display(model_feature_set.tail(3))

,customer_id,time_spent,pages_viewed,basket_value,device_type_Desktop,device_type_Mobile,device_type_Tablet,device_type_Unknown,customer_type_New,customer_type_Returning,purchase
497,998,0.039019,0.333333,0.202147,1,0,0,0,0,1,1
498,999,0.944895,0.888889,0.369052,0,1,0,0,0,1,1
499,1000,0.383350,0.111111,0.523196,0,1,0,0,0,1,1


# Task 3

Now that all preparatory work has been done, create and train a neural network that would allow the company to predict purchases.

- Using PyTorch, create a network with:
   - At least one hidden layer with 8 units
   - ReLU activation for hidden layer
   - Sigmoid activation for the output layer
- Using the prepared features in `input_model_features.csv`, train the model to predict purchases. 
- Use the validation dataset `validation_features.csv` to predict new values based on the trained model. 
- Your model should be named `purchase_model` and your output should be a DataFrame named `validation_predictions` with columns `customer_id` and `purchase`. The `purchase` column must be your predicted values.


In [ ]:
# Write your answer to Task 3 here

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init

from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy

class MyDataset(Dataset):
    def __init__(self, csv_path, is_train):
        super().__init__()
        self.is_train = is_train
        self.raw_df = pd.read_csv(csv_path)
        self.df = self.raw_df.drop(columns="customer_id")
        
        if is_train :
            df = self.df.to_numpy()
            # re-arrage columns : move 'purchase' (3) to the last
            self.data = df[:, [0, 1, 2, 4, 5, 6, 7, 8, 9, 3]]
        else:
            self.data = self.df.to_numpy()
        
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        # convert float64 to torch.float32
        if self.is_train:
            features = torch.tensor(self.data[idx, :-1], dtype=torch.float32) 
            label = torch.tensor(self.data[idx, -1], dtype=torch.float32) 
            return features, label
        else:
            features = torch.tensor(self.data[idx, :], dtype=torch.float32)
            return features

class Net(nn.Module):
    def __init__(self, n_input):
        super().__init__()
        self.fc1 = nn.Linear(n_input, 26)
        self.fc2 = nn.Linear(26, 24)
        self.fc3 = nn.Linear(24, 22)
        self.fc4 = nn.Linear(22, 20)
        
        self.fc5 = nn.Linear(20, 18)
        self.fc6 = nn.Linear(18, 14)
        self.fc7 = nn.Linear(14, 8)
        self.fc8 = nn.Linear(8, 1)

        ### init weight
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight)
        init.kaiming_uniform_(self.fc4.weight)
        
        init.kaiming_uniform_(self.fc5.weight)
        init.kaiming_uniform_(self.fc6.weight)
        init.kaiming_uniform_(self.fc7.weight)
        init.kaiming_uniform_(
            self.fc8.weight,
            nonlinearity="sigmoid",
        )
        
    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.relu(self.fc3(x))
        x = nn.functional.relu(self.fc4(x))
        
        x = nn.functional.relu(self.fc5(x))
        x = nn.functional.relu(self.fc6(x))
        x = nn.functional.relu(self.fc7(x))
        x = nn.functional.sigmoid(self.fc8(x))
        return x

### prepare dataset

dataset_train = MyDataset("input_model_features.csv", is_train=True)
dataloader_train = DataLoader(
    dataset_train, 
    batch_size=2, 
    shuffle=True,
)

### check before training
# features, labels = next(iter(dataloader_train))
# print(f"Features: {features.shape},\nLabels: {labels}")

### start training

purchase_model = Net(n_input=9)
criterion = nn.BCELoss()
optimizer = optim.Adam(purchase_model.parameters(), lr=0.01)

for epoch in range(20):
    training_loss = 0.0
    for features, labels in dataloader_train:
        optimizer.zero_grad()
        outputs = purchase_model(features)
        loss = criterion(
            outputs, labels.view(-1, 1)
        )
        loss.backward()
        optimizer.step()
    
    # Calculate and sum the loss
    # training_loss += loss.item()
    # epoch_loss = training_loss / len(dataloader_train)
    # print(f"Epoch {epoch+1}, Loss: {loss.item()}, Epoch loss {epoch_loss}")

### Evaluation

purchase_model.eval()

dataset_test = MyDataset(csv_path="validation_features.csv", is_train=False)
dataloader_test = DataLoader(dataset_test, batch_size=2, shuffle=False)

predictions = []

with torch.no_grad():
    for features, _ in dataloader_test:

        outputs = purchase_model(features)
        
        predicted_classes = (outputs >= 0.5).float()
        predictions.extend(predicted_classes)


validation_predictions = pd.read_csv("validation_features.csv")
validation_predictions = pd.DataFrame({
    'customer_id': dataset_test.raw_df['customer_id'],
    'prediction': predictions
})

validation_predictions['prediction'] = validation_predictions['prediction'].apply(lambda x: int(x.item()))

ผลการพยากรณ์: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


ValueError: array length 100 does not match index length 200

In [ ]:
validation_predictions

100